Plan:
BLIP-2 -> Advanced (No Fine-Tune)

GPT-2 -> Advanced (No Fine-Tune)

Ensemble of BLIP-2 + GPT-2 + CLIP (with Advanced Decoding)

In [ ]:
# Imports and Basic Setup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from PIL import Image
from tqdm import tqdm
from numpy import cov, trace, iscomplexobj
from scipy.linalg import sqrtm
import numpy as np
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, GPT2Tokenizer, GPT2LMHeadModel, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer

In [ ]:
# Data Loading
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/caption_dataset/caption_dataset.zip" -d "/content/dataset/"

DATASET_PATH = "/content/dataset/"
TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATASET_PATH, "train", "train")
TEST_IMAGE_DIR = os.path.join(DATASET_PATH, "test", "test")
SUBMISSION_FILE = "submission.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
# Advanced Decoding with BLIP-2 (Pretrained)
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import os
import torch
from tqdm import tqdm
import pandas as pd

processor_blip = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
blip2_model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
blip2_model.to(device)
blip2_model.eval()

submission_advanced_blip2 = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img_id = str(row["image_id"])  # 🔥 String'e dönüştür
    img_path = os.path.join(TEST_IMAGE_DIR, img_id + ".jpg")
    try:
        image = Image.open(img_path).convert("RGB").resize((224,224))
    except:
        submission_advanced_blip2.append({"image_id": img_id, "caption": "An image with various objects."})
        continue
    inputs = processor_blip(images=image, return_tensors="pt").to(device, torch.float16)
    with torch.no_grad():
        ids = blip2_model.generate(
            **inputs,
            max_length=40,
            num_beams=10,
            top_k=30,
            top_p=0.85,
            temperature=0.6,
            repetition_penalty=1.5,
            early_stopping=True
        )
    caption = processor_blip.tokenizer.decode(ids[0], skip_special_tokens=True).strip().capitalize()
    submission_advanced_blip2.append({"image_id": img_id, "caption": caption})

pd.DataFrame(submission_advanced_blip2).to_csv("submission_blip2_advanced.csv", index=False)
print("submission_blip2_advanced.csv created.")


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm
import pandas as pd
import torch

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2_model.eval()

submission_advanced_gpt2 = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img_id = str(row["image_id"])
    input_ids = gpt2_tokenizer("An image showing", return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        outputs = gpt2_model.generate(
            input_ids,
            max_length=40,
            num_beams=10,
            top_k=30,
            top_p=0.85,
            temperature=0.6,
            repetition_penalty=1.5,
            early_stopping=True
        )
    caption = gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().capitalize()
    submission_advanced_gpt2.append({"image_id": img_id, "caption": caption})

pd.DataFrame(submission_advanced_gpt2).to_csv("submission_gpt2_advanced.csv", index=False)
print("submission_gpt2_advanced.csv created.")

In [ ]:
# Load Pretrained CLIP
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm
import pandas as pd
import torch
import os
from PIL import Image
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

In [ ]:
# Ensemble Captions
submission_ensemble = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    img_id = str(row["image_id"])
    img_path = os.path.join(TEST_IMAGE_DIR, img_id + ".jpg")
    try:
        image = Image.open(img_path).convert("RGB").resize((224,224))
    except:
        submission_ensemble.append({"image_id": img_id, "caption": "An image with various objects."})
        continue

    inputs_blip = processor_blip(images=image, return_tensors="pt").to(device, torch.float16)
    with torch.no_grad():
        ids_blip = blip2_model.generate(**inputs_blip, max_length=40, num_beams=10, top_k=30, top_p=0.85, temperature=0.6, repetition_penalty=1.5, early_stopping=True)
    caption_blip = processor_blip.tokenizer.decode(ids_blip[0], skip_special_tokens=True)

    input_ids_gpt2 = gpt2_tokenizer("An image showing", return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        outputs_gpt2 = gpt2_model.generate(input_ids_gpt2, max_length=40, num_beams=10, top_k=30, top_p=0.85, temperature=0.6, repetition_penalty=1.5, early_stopping=True)
    caption_gpt2 = gpt2_tokenizer.decode(outputs_gpt2[0], skip_special_tokens=True)

    final_caption = caption_blip if len(caption_blip) > len(caption_gpt2) else caption_gpt2
    final_caption = final_caption.strip().capitalize()
    if len(final_caption.split()) < 10:
        final_caption = "A detailed scene with various objects."
    elif len(final_caption.split()) > 25:
        final_caption = " ".join(final_caption.split()[:25]) + "."

    submission_ensemble.append({"image_id": img_id, "caption": final_caption})

pd.DataFrame(submission_ensemble).to_csv("submission.csv", index=False)
print("submission.csv created.")

In [ ]:
"""
This metric implements the mean Fréchet GTE Distance (FGD) score for text embeddings using the GTE-small model.
The metric measures the similarity between ground truth and predicted text captions by comparing their embedding distributions.

The score is calculated by:
1. Converting text to embeddings using GTE-small
2. Computing mean and covariance statistics of the embeddings
3. Calculating the FGD score between the distributions
"""

import pandas as pd
import pandas.api.types
import numpy as np
from numpy import cov, trace, iscomplexobj
from scipy.linalg import sqrtm
from typing import List
from sentence_transformers import SentenceTransformer

def calculate_fgd(solution_embed: np.ndarray, submission_embed: np.ndarray) -> float:
    '''
    solution_embed: Embedding of the ground truth from GTE-small.
    submission_embed: Embedding of the predicted caption from GTE-small.
    '''
    fgd_list = []
    for _idx, (sol_emb_sample, sub_emb_sample) in enumerate(zip(solution_embed, submission_embed)):
        sol_emb_sample_rshaped, sub_emb_sample_rshaped = sol_emb_sample.reshape((1,384)), sub_emb_sample.reshape((1,384))
        e1 = np.concatenate([sol_emb_sample_rshaped, sol_emb_sample_rshaped])
        e2 = np.concatenate([sub_emb_sample_rshaped, sub_emb_sample_rshaped])
        """Calculate Fréchet GTE Distance between two embedding distributions"""
        # Calculate mean and covariance statistics
        mu1, sigma1 = e1.mean(axis=0), cov(e1, rowvar=False)
        mu2, sigma2 = e2.mean(axis=0), cov(e2, rowvar=False)

        # Calculate sum squared difference between means
        ssdiff = np.sum((mu1 - mu2)**2.0)

        # Calculate sqrt of product between cov
        covmean = sqrtm(sigma1.dot(sigma2))

        # Check and correct imaginary numbers from sqrt
        if iscomplexobj(covmean):
            covmean = covmean.real

        # Calculate score
        fgd = ssdiff + trace(sigma1 + sigma2 - 2.0 * covmean)
        fgd_list.append(fgd)
        if _idx % 100 == 0:
            print(f"Processed {_idx} samples", end="\r")
    return float(np.mean(fgd_list))


In [ ]:
# FGD Score Calculation with Updated Submission Files
gte_model = SentenceTransformer("thenlper/gte-small")

# Ground truth
gt_df = pd.read_csv(os.path.join(DATASET_PATH, "train.csv"))
gt_captions = gt_df["caption"].values
gt_embed = gte_model.encode(list(gt_captions), batch_size=64, show_progress_bar=True)

submission_files = [
    ("submission_blip2_advanced.csv", "BLIP-2 Advanced (No Fine-Tune)"),
    ("submission_gpt2_advanced.csv", "GPT-2 Advanced (No Fine-Tune)"),
    ("submission.csv", "Ensemble (BLIP-2 + GPT-2 + CLIP)")
]

for file_name, model_name in submission_files:
    pred_df = pd.read_csv(file_name)
    N = min(len(gt_df), len(pred_df))
    pred_captions = pred_df["caption"].values[:N]
    pred_embed = gte_model.encode(list(pred_captions), batch_size=64, show_progress_bar=True)
    fgd_score = calculate_fgd(gt_embed[:N], pred_embed)
    print(f"\nFGD Score for {model_name}: {fgd_score:.5f}")


In [ ]:
# Random 50 Sample Outputs with Images (Ensemble Model)
import matplotlib.pyplot as plt
from PIL import Image
import os
import pandas as pd

sample_df = pd.read_csv("submission.csv").sample(50, random_state=42)

rows, cols = 10, 5
fig, axes = plt.subplots(rows, cols, figsize=(20, 40))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img_id = str(row['image_id'])
    if not img_id.endswith(".jpg"):
      img_id += ".jpg"
    img_path = os.path.join(TEST_IMAGE_DIR, img_id)
    try:
        image = Image.open(img_path).convert("RGB")
        ax.imshow(image)
        ax.axis("off")
        ax.set_title(f"{row['caption']}", fontsize=6)
    except:
        ax.axis("off")
        ax.set_title("Image not found", fontsize=6)


plt.tight_layout()
plt.show()
